In [1]:
from sentence_transformers import SentenceTransformer

import numpy as np

C:\Users\gaand\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import pandas as pd
from pathlib import Path

carpeta = Path("../data/processed")
df = pd.concat(
    [pd.read_csv(f) for f in sorted(carpeta.glob("chunks_*.csv"))],
    ignore_index=True,
)

df["chunk_index"] = range(len(df))

documentos = df["contenido"].tolist()
print("documentos:", len(documentos))

documentos: 89


In [13]:
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

embedding_docs = modelo.encode(documentos)

print(embedding_docs.shape)  # debe dar (n, 384)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1641.98it/s]


(89, 384)


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
def buscar_documentos_similares(consulta, documentos, embedding_docs, modelo, top_k=3):

    # 1. Convierte la pregunta a embedding
    embedding_consulta = modelo.encode([consulta])

    # 2. Calcula la similitud entre todos los documentos y la consulta
    similitudes = cosine_similarity(embedding_consulta, embedding_docs)[0]

    # 3. Obtén los índices de los documentos más similares
    indices_top = np.argsort(similitudes)[::-1][:top_k]

    # 4. Devuelve los documentos más similares y sus puntuaciones de similitud
    resultados = [(documentos[i], similitudes[i]) for i in indices_top]

    return resultados

In [24]:
pregunta = "¿puedo ver gratis lo que mi banco reporta a la Central de Información Crediticia?"

resultados = buscar_documentos_similares(pregunta, documentos, embedding_docs, modelo, top_k=3)

for texto, score in resultados:
    print(f"Documento: {texto}\nSimilitud: {score:.4f}\n")

Documento: Lineamientos para la prevención de operaciones no reconocidas con tarjetas, artículo 13: INVESTIGACIÓN DE RECLAMOS POR TRANSACCIONES MONETARIAS NO RECONOCIDAS Las Instituciones Supervisadas deben realizar una investigación exhaustiva de todos los reclamos por transacciones monetarias no reconocidas presentadas por los tarjetahabientes, la cual debe estar debidamente documentada y disponible en caso de ser requerida por la CNBS.
Similitud: 0.5708

Documento: Obligaciones de las instituciones supervisadas por la CNBS: Permitir al usuario la consulta gratuita y por los canales que la institución supervisada establezca, en relación a: Estado de sus cuentas, créditos, servicios y productos, incluyendo la información que la institución supervisada reporta a la Central de Información Crediticia (CIC).
Similitud: 0.5536

Documento: Lineamientos para la prevención de operaciones no reconocidas con tarjetas, artículo 12: PROCESO DE RECLAMO SOBRE TRANSACCIONES MONETARIAS NO RECONOCIDAS

In [25]:

np.save("../data/processed/embeddings.npy", embedding_docs)
print("guardado:", embedding_docs.shape)

guardado: (89, 384)
